# Кандидатогенерация для поиска услуг Авито

**Самостоятельный ноутбук:** для запуска нужны только этот `.ipynb` и три
исходных Parquet-файла. Весь код обработки, обучения, оценки и инференса
находится ниже. Папка `src`, Python-скрипты и готовая модель не нужны.

Положите данные в `datasets/` рядом с ноутбуком (либо измените `DATA_DIR`),
выберите Python 3.12 и выполните **Restart Kernel → Run All**. В Colab можно
использовать CPU; GPU для этой версии не нужен. На выходе получатся `answer.csv`,
сохранённая модель и отчёты в `notebook_outputs/`.

Цель — вернуть 50 уникальных объявлений на запрос. Метрика — средний по
запросам Recall@50. Начинаем с быстрого локального поиска: BM25, символьный
TF-IDF и мягкие признаки географии и подкатегории.

## 1. Окружение и настройки

Ячейка ниже устанавливает зависимости только при отсутствии нужной библиотеки.
Для точного повторения версий можно включить `INSTALL_EXACT_VERSIONS` и после
установки перезапустить ядро. Установка пакетов может обращаться к PyPI;
дальнейшие обучение и поиск работают полностью локально.

In [1]:
import importlib.util
import importlib.metadata
import subprocess
import sys

INSTALL_EXACT_VERSIONS = False
PACKAGES = {
    'numpy': ('numpy', '2.5.3'),
    'scipy': ('scipy', '1.18.1'),
    'pandas': ('pandas', '3.0.6'),
    'pyarrow': ('pyarrow', '25.0.1'),
    'sklearn': ('scikit-learn', '1.9.1'),
    'nltk': ('nltk', '3.10.3'),
    'tqdm': ('tqdm', '4.70.1'),
    'cloudpickle': ('cloudpickle', '3.1.2'),
}
missing = [name for name in PACKAGES if importlib.util.find_spec(name) is None]
if missing or INSTALL_EXACT_VERSIONS:
    requirements = [f'{package}=={version}' for package, version in PACKAGES.values()]
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *requirements])
print('Python:', sys.version.split()[0])
print({package: importlib.metadata.version(package) for package, _ in PACKAGES.values()})


Python: 3.12.13
{'numpy': '2.5.3', 'scipy': '1.18.1', 'pandas': '3.0.6', 'pyarrow': '25.0.1', 'scikit-learn': '1.9.1', 'nltk': '3.10.3', 'tqdm': '4.70.1', 'cloudpickle': '3.1.2'}


In [2]:
from pathlib import Path
import gc
import gzip
import hashlib
import json
import logging
import time

import cloudpickle
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

DATA_DIR = Path('datasets')
OUTPUT_DIR = Path('.')
SEED = 2026
VALIDATION_QUERIES = 1200

EXPECTED_FILES = ['train.parquet', 'benchmark_queries.parquet', 'benchmark_items.parquet']
if not DATA_DIR.exists() and all(Path(name).exists() for name in EXPECTED_FILES):
    DATA_DIR = Path('.')
for name in EXPECTED_FILES:
    assert (DATA_DIR / name).is_file(), f'Не найден {DATA_DIR / name}; проверь DATA_DIR'

REPORT_DIR = OUTPUT_DIR / 'notebook_outputs'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = REPORT_DIR / 'model.pkl.gz'
ANSWER_PATH = OUTPUT_DIR / 'answer.csv'
previous_answer_hash = hashlib.sha256(ANSWER_PATH.read_bytes()).hexdigest() if ANSWER_PATH.exists() else None
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
QUERY_COLUMNS = ['search_query', 'search_location_id', 'search_is_delivery_search',
                 'search_infm_params_text', 'search_category']
started = time.perf_counter()


## 2. Обработка текста и чтение данных

Идентификаторы сохраняем строками. Нормализуем только тексты: регистр, `ё/е`,
HTML, пунктуацию. Стемминг Snowball объединяет часть русских словоформ;
скачивание словарей не требуется. Кэш ускоряет обработку повторяющихся слов.

В train есть только положительные пары. Запрос определяем по всем пяти
`search_*` полям: одинаковый текст в другом городе или с другими фильтрами
может иметь другой набор позитивов.

In [3]:
"""Обработка русского текста без загрузки словарей или внешних сервисов."""

import html
import re

from nltk.stem.snowball import RussianStemmer

_STEMMER = RussianStemmer()
_WORDS = re.compile(r"[a-zа-я0-9]+")
_HTML = re.compile(r"<[^>]+>")
_STOP = set("и в во на по для с со из от до за к ко у о об а но или это как мы вы я он она они наш ваш при без под над не что же ли бы все еще уже так то же".split())


def normalize(text: str) -> str:
    """Осторожная нормализация; идентификаторы через эту функцию не проходят."""
    text = html.unescape(str(text or "")).lower().replace("ё", "е")
    return " ".join(_WORDS.findall(_HTML.sub(" ", text)))


# Обычный словарь сохраняется вместе с функцией через cloudpickle.
# В отличие от lru_cache, он не требует определения stem в новом процессе.
_STEM_CACHE = {}


def stem(word: str) -> str:
    if word not in _STEM_CACHE:
        result = _STEMMER.stem(word)
        if len(_STEM_CACHE) < 300_000:
            _STEM_CACHE[word] = result
        return result
    return _STEM_CACHE[word]


def tokenize(text: str) -> list[str]:
    return [stem(w) for w in normalize(text).split() if len(w) > 1 and w not in _STOP]


In [4]:
"""Загрузка данных и разделение по текстам запросов без утечки позитивов."""

from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq


TRAIN_COLUMNS = QUERY_COLUMNS + ["item_id", "item_microcat_id", "item_location_id",
                                  "item_latitude", "item_longitude"]


def read_parquet(path, columns=None, filters=None):
    table = pq.read_table(path, columns=columns, filters=filters)
    # Decimal нужен исходному хранилищу, но для координат/цены достаточно float64.
    for i, field in enumerate(table.schema):
        if pa.types.is_decimal(field.type):
            table = table.set_column(i, field.name, table.column(i).cast(pa.float64()))
    return table.to_pandas()


def validation_split(train, n_queries=1200, seed=2026):
    normalized = train.search_query.map(normalize)
    texts = np.array(sorted(normalized.unique()))
    if n_queries >= len(texts):
        raise ValueError("Для обучения должны остаться тексты запросов")
    rng = np.random.default_rng(seed)
    chosen = rng.choice(texts, size=n_queries, replace=False)
    heldout = train.loc[normalized.isin(chosen)]
    contexts = heldout[QUERY_COLUMNS].drop_duplicates().copy()
    contexts["normalized_query"] = contexts.search_query.map(normalize)
    contexts["random_order"] = rng.random(len(contexts))
    # Одна комбинация контекста на текст: иначе частые формулировки подавят редкие.
    selected = contexts.sort_values("random_order").drop_duplicates("normalized_query")
    selected = selected.sort_values("normalized_query").reset_index(drop=True)
    labels = heldout.groupby(QUERY_COLUMNS, dropna=False).item_id.agg(lambda ids: sorted(set(ids)))
    selected["relevant_ids"] = [labels.loc[tuple(row)] for row in selected[QUERY_COLUMNS].itertuples(index=False, name=None)]
    order = rng.permutation(len(selected))
    selected["split"] = "holdout"
    selected.loc[order[:int(len(selected) * 2 / 3)], "split"] = "dev"
    selected["query_id"] = [f"validation_{i:05d}" for i in range(len(selected))]
    selected = selected.drop(columns="random_order")
    fitting = train.loc[~normalized.isin(chosen)].copy()
    assert not set(fitting.search_query.map(normalize)) & set(selected.normalized_query)
    return fitting, selected


def load_training_problem(data_dir: Path, n_queries=1200, seed=2026):
    train = read_parquet(data_dir / "train.parquet", columns=TRAIN_COLUMNS)
    corpus = read_parquet(data_dir / "benchmark_items.parquet")
    queries = read_parquet(data_dir / "benchmark_queries.parquet")
    fitting, validation = validation_split(train, n_queries=n_queries, seed=seed)
    relevant_ids = set(i for ids in validation.relevant_ids for i in ids)
    absent = sorted(relevant_ids - set(corpus.item_id))
    extra = read_parquet(data_dir / "train.parquet", columns=list(corpus.columns),
                         filters=[("item_id", "in", absent)]) if absent else corpus.iloc[:0]
    # Если объявление уже есть в benchmark, его версию считаем канонической.
    index_items = pd.concat([corpus, extra], ignore_index=True).drop_duplicates("item_id")
    assert relevant_ids <= set(index_items.item_id)
    return train, fitting, validation, corpus, index_items, queries


In [5]:
files = []
for name in EXPECTED_FILES:
    path = DATA_DIR / name
    parquet = pq.ParquetFile(path)
    files.append({'file': name, 'rows': parquet.metadata.num_rows,
                  'columns': len(parquet.schema_arrow),
                  'MiB': round(path.stat().st_size / 2**20, 1)})
display(pd.DataFrame(files))

train, fitting, validation, corpus, index_items, queries = load_training_problem(
    DATA_DIR, n_queries=VALIDATION_QUERIES, seed=SEED)

eda = {
    'Уникальные контексты train': len(train[QUERY_COLUMNS].drop_duplicates()),
    'Уникальные нормализованные тексты train': train.search_query.map(normalize).nunique(),
    'Повторные пары контекст–объявление': int(train.duplicated(QUERY_COLUMNS + ['item_id']).sum()),
    'Доля пар с разными локациями': float((train.search_location_id != train.item_location_id).mean()),
    'Доля train-строк с item_id в benchmark': float(train.item_id.isin(corpus.item_id).mean()),
}
display(pd.Series(eda, name='value').to_frame())


,file,rows,columns,MiB
0,train.parquet,497673,19,467.5
1,benchmark_queries.parquet,2452,6,0.1
2,benchmark_items.parquet,189212,14,186.6


,value
Уникальные контексты train,354463.000000
Уникальные нормализованные тексты train,73706.000000
Повторные пары контекст–объявление,30619.000000
Доля пар с разными локациями,0.168980
Доля train-строк с item_id в benchmark,0.066329


## 3. Валидация

Разделение выполнено **по нормализованным текстам**: все пары с отложенными
формулировками удалены из обучения статистик. На каждый текст берём один
случайный контекст и все известные позитивы этого контекста. Повторы
позитивных `item_id` удаляем.

Из 1 200 запросов 800 используются для выбора конфигурации, 400 — только
для итоговой проверки. Все позитивные объявления должны присутствовать
в поисковом корпусе. Поэтому к benchmark добавлены недостающие документы
из train. Индекс видит их тексты, но не validation-связи запрос–объявление.

**Локальная оценка не равна метрике скрытого бенчмарка.** Состав конкурирующих
объявлений и распределение текстов отличаются; это может делать оценку
оптимистичной. Bootstrap-интервал ниже не учитывает этот сдвиг.

In [6]:
assert not set(fitting.search_query.map(normalize)) & set(validation.normalized_query)
assert validation.normalized_query.nunique() == len(validation)
assert set(i for ids in validation.relevant_ids for i in ids) <= set(index_items.item_id)
display(validation.groupby('split').agg(queries=('query_id', 'size')))
print('Документов в корпусе валидации:', len(index_items))
print('Пар для обучения статистик:', len(fitting))
validation.to_parquet(REPORT_DIR / 'validation_queries.parquet', index=False)


,queries
split,
dev,800
holdout,400


Документов в корпусе валидации: 190392
Пар для обучения статистик: 489371


## 4. Поиск кандидатов

В следующих двух ячейках находится **полная реализация** алгоритма.

* BM25 отдельно по заголовку и по заголовку + параметрам + описанию.
  Параметры обрезаются до 1 800 символов, описание — до 5 000.
* Символьные n-граммы 3–5 по заголовкам помогают сопоставлять словоформы
  и похожие написания. Сходство — косинусное по TF-IDF.
* География учитывается мягко: совпадение локации, близость координат
  и переходы между поисковой локацией и локацией объявления из train.
* Для прогноза подкатегории используем 20 похожих train-запросов и
  распределения выбранных ими `item_microcat_id`, с весами similarity^6.
* Рейтинг и число отзывов дают небольшую добавку качества.

`score = text_score * (1 + w_geo * geo) * (1 + w_category * category) + w_quality * quality`

Жёстких географических исключений нет. При равных оценках используем порядок
по `item_id`, чтобы top-50 воспроизводился. Не используем ручные ответы
или правила под конкретные benchmark `query_id`.

In [7]:
"""Разреженный текстовый поиск и мягкие признаки контекста.

BM25 вычисляется как произведение разреженных матриц: вся коллекция
не обходится Python-циклом для каждого запроса. Все обучаемые статистики
сосредоточены в fit_history(), чтобы явно исключать validation из обучения.
"""

from dataclasses import asdict, dataclass
import logging

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import normalize as l2_normalize
from tqdm.auto import tqdm


LOG = logging.getLogger(__name__)
QUERY_COLUMNS = ["search_query", "search_location_id", "search_is_delivery_search",
                 "search_infm_params_text", "search_category"]
FEATURES = ["title", "body", "char", "filters", "geo", "microcat", "quality"]


@dataclass
class SearchConfig:
    title: float = 1.5
    body: float = 1.0
    char: float = 1.0
    filters: float = 0.15
    geo: float = 3.0
    microcat: float = 0.5
    quality: float = 0.05

    def as_dict(self):
        return asdict(self)


def topk(scores: np.ndarray, k: int) -> np.ndarray:
    """Детерминированный top-k: при равных оценках меньший индекс раньше.

    Документы заранее сортируются по item_id. Учитываем также равенство
    оценок на границе top-k, которое один argpartition не разрешает стабильно.
    """
    k = min(k, len(scores))
    if k == 0:
        return np.empty(0, dtype=np.int32)
    cutoff = np.partition(scores, len(scores) - k)[len(scores) - k]
    above = np.flatnonzero(scores > cutoff)
    tied = np.flatnonzero(scores == cutoff)[:k - len(above)]
    selected = np.concatenate([above, tied])
    return selected[np.lexsort((selected, -scores[selected]))].astype(np.int32)


class BM25Field:
    def __init__(self, max_features=250_000, k1=1.2, b=0.65):
        self.k1, self.b = k1, b
        self.vectorizer = CountVectorizer(
            tokenizer=tokenize, token_pattern=None, lowercase=False,
            min_df=2, max_features=max_features, dtype=np.float32,
        )

    def fit(self, documents):
        counts = self.vectorizer.fit_transform(documents).tocsr()
        n = counts.shape[0]
        lengths = np.asarray(counts.sum(axis=1)).ravel()
        df = np.bincount(counts.indices, minlength=counts.shape[1])
        self.idf = np.log1p((n - df + 0.5) / (df + 0.5)).astype(np.float32)
        norm = self.k1 * (1 - self.b + self.b * lengths / max(lengths.mean(), 1))
        denominator = counts.data + np.repeat(norm, np.diff(counts.indptr))
        counts.data = (counts.data * (self.k1 + 1) / denominator
                       * self.idf[counts.indices]).astype(np.float32)
        self.postings = counts.T.tocsr()
        return self

    def transform_queries(self, texts):
        q = self.vectorizer.transform(texts).tocsr()
        q.data[:] = 1  # Повтор слова в запросе не усиливает его искусственно.
        denominator = np.asarray(q @ self.idf).ravel() * (self.k1 + 1)
        return q, np.maximum(denominator, 1e-6)

    def score(self, row, denominator):
        return (row @ self.postings).toarray().ravel() / denominator

    def restrict(self, indices):
        self.postings = self.postings[:, indices].tocsr()


/Users/strafe/VSCode-Projects/DataScience-Bootcamp2026/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
class CandidateModel:
    def fit_index(self, items: pd.DataFrame):
        """Индекс использует только тексты и метаданные документов, не метки."""
        items = items.drop_duplicates("item_id").sort_values("item_id").reset_index(drop=True)
        self.item_ids = items.item_id.to_numpy(dtype=str)
        self.locations = items.item_location_id.fillna(-1).to_numpy(np.int64)
        self.microcats = items.item_microcat_id.fillna(-1).to_numpy(np.int64)
        self.latitudes = pd.to_numeric(items.item_latitude, errors="coerce").to_numpy(float)
        self.longitudes = pd.to_numeric(items.item_longitude, errors="coerce").to_numpy(float)
        reviews = items.item_rating_reviews_count.fillna(0).to_numpy(np.float32)
        ratings = items.item_rating.fillna(0).to_numpy(np.float32)
        self.quality = (0.75 * np.minimum(np.log1p(np.maximum(reviews, 0)) / 7, 1)
                        + 0.25 * np.clip(ratings / 5, 0, 1)).astype(np.float32)
        titles = items.item_title_raw.fillna("").tolist()
        bodies = (items.item_title_raw.fillna("") + " "
                  + items.item_infm_params_text.fillna("").str.slice(0, 1800) + " "
                  + items.item_description_raw.fillna("").str.slice(0, 5000)).tolist()
        LOG.info("BM25: заголовки, %s документов", len(items))
        self.title_index = BM25Field(max_features=100_000, b=0.35).fit(titles)
        LOG.info("BM25: описание и параметры")
        self.body_index = BM25Field().fit(bodies)
        LOG.info("TF-IDF: символьные n-граммы заголовков")
        self.char_vectorizer = TfidfVectorizer(
            preprocessor=normalize, analyzer="char_wb", ngram_range=(3, 5),
            min_df=2, max_features=180_000, sublinear_tf=True, dtype=np.float32,
        )
        self.char_postings = self.char_vectorizer.fit_transform(titles).T.tocsr()
        self.config = SearchConfig()
        return self

    def fit_history(self, train: pd.DataFrame, location_metadata: pd.DataFrame):
        """Обучение географических переходов и kNN-классификатора подкатегории.

        В kNN хранятся текст запроса и распределение подкатегорий выбранных
        объявлений. Связи с query_id бенчмарка нигде не используются.
        """
        LOG.info("Статистики контекста: %s обучающих пар", len(train))
        unique = train.drop_duplicates(QUERY_COLUMNS + ["item_id"])
        geo = unique.groupby(["search_location_id", "item_location_id"]).size()
        self.geo_routes = {}
        for search_loc, group in geo.groupby(level=0):
            counts = group.droplevel(0)
            # Мягкая поправка помогает при поиске по региону/соседнему городу.
            affinity = np.sqrt(counts / counts.max())
            self.geo_routes[int(search_loc)] = affinity.to_dict()
        coords = location_metadata[["item_location_id", "item_latitude", "item_longitude"]].copy()
        for c in ["item_latitude", "item_longitude"]:
            coords[c] = pd.to_numeric(coords[c], errors="coerce")
        coords = coords.dropna().drop_duplicates()
        coords = coords[(coords.item_latitude.between(-90, 90))
                        & coords.item_longitude.between(-180, 180)]
        centers = coords.groupby("item_location_id")[["item_latitude", "item_longitude"]].median()
        self.location_centers = {int(k): tuple(v) for k, v in zip(centers.index, centers.to_numpy())}

        qdata = unique[["search_query", "item_microcat_id"]].copy()
        qdata["normalized"] = qdata.search_query.map(normalize)
        grouped = qdata.groupby(["normalized", "item_microcat_id"]).size().reset_index(name="count")
        self.history_texts = np.array(sorted(grouped.normalized.unique()), dtype=str)
        self.category_values = np.array(sorted(grouped.item_microcat_id.unique()), dtype=np.int64)
        qr = pd.Index(self.history_texts).get_indexer(grouped.normalized)
        cr = pd.Index(self.category_values).get_indexer(grouped.item_microcat_id)
        label_matrix = sparse.csr_matrix(
            (grouped["count"].to_numpy(np.float32), (qr, cr)),
            shape=(len(self.history_texts), len(self.category_values)),
        )
        self.history_labels = l2_normalize(label_matrix, norm="l1", axis=1)
        self.history_vectorizer = TfidfVectorizer(
            analyzer="char_wb", ngram_range=(3, 5), min_df=2,
            max_features=100_000, sublinear_tf=True, dtype=np.float32,
        )
        self.history_postings = self.history_vectorizer.fit_transform(self.history_texts).T.tocsr()
        self.item_category_indices = pd.Index(self.category_values).get_indexer(self.microcats)
        return self

    def geo_scores(self, location):
        same = (self.locations == location).astype(np.float32)
        scores = same.copy()
        center = self.location_centers.get(int(location))
        if center is not None:
            lat1, lon1 = np.radians(center)
            lat2, lon2 = np.radians(self.latitudes), np.radians(self.longitudes)
            a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
            distance = 6371.0088 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
            near = np.nan_to_num(np.exp(-distance / 40), nan=0).astype(np.float32)
            scores = np.maximum(scores, 0.7 * near)
        routes = self.geo_routes.get(int(location), {})
        if routes:
            locations, inverse = np.unique(self.locations, return_inverse=True)
            affinity = np.array([routes.get(int(loc), 0) for loc in locations], dtype=np.float32)
            scores = np.maximum(scores, 0.65 * affinity[inverse])
        return scores

    def category_scores(self, query_vector):
        similarities = (query_vector @ self.history_postings).toarray().ravel()
        neighbors = topk(similarities, 20)
        weights = similarities[neighbors] ** 6
        probs = np.asarray(self.history_labels[neighbors].T @ weights).ravel()
        if probs.max(initial=0) > 0:
            probs /= probs.max()
        scores = np.zeros(len(self.item_ids), dtype=np.float32)
        known = self.item_category_indices >= 0
        scores[known] = probs[self.item_category_indices[known]]
        return scores

    @staticmethod
    def combine(features, config):
        text = (config.title * features[:, 0] + config.body * features[:, 1]
                + config.char * features[:, 2] + config.filters * features[:, 3])
        # География/категория усиливают текстовое совпадение, но не отсекают
        # объявления и не могут полностью заменить совпадение текста.
        return (text * (1 + config.geo * features[:, 4])
                * (1 + config.microcat * features[:, 5])
                + config.quality * features[:, 6])

    def iter_features(self, queries, progress=True):
        texts = queries.search_query.fillna("").tolist()
        filters = queries.search_infm_params_text.fillna("").tolist()
        tq, td = self.title_index.transform_queries(texts)
        bq, bd = self.body_index.transform_queries(texts)
        fq, fd = self.body_index.transform_queries(filters)
        cq = self.char_vectorizer.transform(texts)
        hq = self.history_vectorizer.transform([normalize(t) for t in texts])
        # В кэше только локации текущего пакета запросов; после вызова он удаляется.
        geo_cache = {}
        for i, row in enumerate(tqdm(queries.itertuples(index=False), total=len(queries),
                                     disable=not progress, desc="Поиск")):
            loc = int(row.search_location_id)
            if loc not in geo_cache:
                if len(geo_cache) >= 32:
                    geo_cache.clear()
                geo_cache[loc] = self.geo_scores(loc)
            yield np.column_stack([
                self.title_index.score(tq[i], td[i]),
                self.body_index.score(bq[i], bd[i]),
                (cq[i] @ self.char_postings).toarray().ravel(),
                self.body_index.score(fq[i], fd[i]),
                geo_cache[loc], self.category_scores(hq[i]), self.quality,
            ]).astype(np.float32, copy=False)

    def predict(self, queries, k=50, progress=True):
        return [self.item_ids[topk(self.combine(f, self.config), k)].tolist()
                for f in self.iter_features(queries, progress=progress)]

    def restrict_items(self, allowed_ids):
        """Финальный индекс возвращает только документы разрешённого корпуса.

        Словарь/IDF сохраняются: они обучены на текстах benchmark и добавленных
        документах train, без использования validation-меток в скоринге.
        """
        keep = np.flatnonzero(np.isin(self.item_ids, list(allowed_ids)))
        for name in ["item_ids", "locations", "microcats", "latitudes", "longitudes", "quality",
                     "item_category_indices"]:
            setattr(self, name, getattr(self, name)[keep])
        self.title_index.restrict(keep)
        self.body_index.restrict(keep)
        self.char_postings = self.char_postings[:, keep].tocsr()
        return self


In [9]:
location_metadata = pd.concat([
    train[['item_location_id', 'item_latitude', 'item_longitude']],
    corpus[['item_location_id', 'item_latitude', 'item_longitude']],
])
# Здесь только метаданные документов; связи validation-запросов не используются.
model = CandidateModel().fit_index(index_items)
model.fit_history(fitting, location_metadata)
print('Индекс готов:', len(model.item_ids), 'документов')


2026-09-28 18:39:30,302 INFO BM25: заголовки, 190392 документов


2026-09-28 18:39:31,239 INFO BM25: описание и параметры


2026-09-28 18:39:59,380 INFO TF-IDF: символьные n-граммы заголовков


2026-09-28 18:40:02,560 INFO Статистики контекста: 489371 обучающих пар


Индекс готов: 190392 документов


## 5. Recall@50, выбор конфигурации и holdout

Для запроса считаем `число найденных позитивов / число всех известных позитивов`.
Затем усредняем по запросам, а не по строкам train. Конфигурацию выбираем
по dev; holdout не участвует в выборе параметров. Ниже сравниваем также
чистый BM25 и комбинацию текстовых каналов без географии.

In [10]:
"""Recall@50 с усреднением по запросам, dev-подбор и отдельный holdout."""

import logging

import numpy as np
import pandas as pd


LOG = logging.getLogger(__name__)


def configs_to_compare():
    configs = {
        "bm25": SearchConfig(char=0, filters=0, geo=0, microcat=0, quality=0),
        "text": SearchConfig(geo=0, microcat=0, quality=0),
    }
    for geo in [2.0, 5.0, 10.0]:
        for cat in [0.0, 0.7, 2.0]:
            configs[f"geo{geo:g}_cat{cat:g}"] = SearchConfig(geo=geo, microcat=cat)
    for char in [0.0, 2.0]:
        configs[f"char{char:g}"] = SearchConfig(char=char, geo=5.0, microcat=0.7)
    configs["no_quality"] = SearchConfig(geo=5, microcat=0.7, quality=0)
    configs["quality_high"] = SearchConfig(geo=5, microcat=0.7, quality=0.25)
    return configs


def evaluate_configs(model, queries, configs):
    item_lookup = pd.Index(model.item_ids)
    truth_indices = [item_lookup.get_indexer(ids) for ids in queries.relevant_ids]
    if any((x < 0).any() or len(x) == 0 for x in truth_indices):
        raise ValueError("Не все validation-позитивы присутствуют в поисковом индексе")
    recalls = {name: [] for name in configs}
    selected_results = {name: [] for name in configs}
    for truth, features in zip(truth_indices, model.iter_features(queries)):
        for name, config in configs.items():
            indices = topk(model.combine(features, config), 50)
            recalls[name].append(float(np.isin(truth, indices).sum() / len(truth)))
            selected_results[name].append(indices)
    return {k: np.array(v) for k, v in recalls.items()}, selected_results


def bootstrap_interval(values, seed=2026):
    rng = np.random.default_rng(seed)
    means = np.mean(rng.choice(values, size=(2000, len(values)), replace=True), axis=1)
    return np.quantile(means, [0.025, 0.975]).tolist()


def tune_and_evaluate(model, validation):
    configs = configs_to_compare()
    dev = validation[validation.split == "dev"].copy()
    holdout = validation[validation.split == "holdout"].copy()
    LOG.info("Подбор %s конфигураций на %s dev-запросах", len(configs), len(dev))
    dev_recalls, _ = evaluate_configs(model, dev, configs)
    scores = {name: float(values.mean()) for name, values in dev_recalls.items()}
    best_name = max(scores, key=scores.get)
    model.config = configs[best_name]
    LOG.info("Лучшая конфигурация %s, dev Recall@50=%.4f", best_name, scores[best_name])
    # Конфигурация фиксируется ДО чтения holdout-метрик.
    holdout_configs = {"bm25": configs["bm25"], "text": configs["text"], "selected": model.config}
    holdout_recalls, predictions = evaluate_configs(model, holdout, holdout_configs)
    metrics = {
        "metric": "macro Recall@50", "split": "grouped by normalized search_query",
        "dev_queries": len(dev), "holdout_queries": len(holdout),
        "validation_corpus_size": len(model.item_ids),
        "dev_scores": scores, "selected_config_name": best_name,
        "selected_config": model.config.as_dict(),
        "holdout_scores": {k: float(v.mean()) for k, v in holdout_recalls.items()},
        "holdout_selected_bootstrap_95pct": bootstrap_interval(holdout_recalls["selected"]),
        "note": "Local validation, not the hidden benchmark score. All held-out positives are in the corpus.",
    }
    details = holdout.copy()
    details["recall_at_50"] = holdout_recalls["selected"]
    details["predicted_ids"] = [model.item_ids[p].tolist() for p in predictions["selected"]]
    details["relevant_count"] = details.relevant_ids.map(len)
    return metrics, details


In [11]:
metrics, details = tune_and_evaluate(model, validation)
display(pd.Series(metrics['dev_scores'], name='dev Recall@50').sort_values(ascending=False).to_frame())
print('Выбранная конфигурация:', metrics['selected_config_name'])
display(pd.Series(metrics['selected_config'], name='weight').to_frame())
display(pd.Series(metrics['holdout_scores'], name='holdout Recall@50').to_frame())
print('Bootstrap 95%:', metrics['holdout_selected_bootstrap_95pct'])
(REPORT_DIR / 'metrics.json').write_text(json.dumps(metrics, ensure_ascii=False, indent=2))
details.to_parquet(REPORT_DIR / 'holdout_details.parquet', index=False)


2026-09-28 18:40:04,225 INFO Подбор 15 конфигураций на 800 dev-запросах



Поиск:   0%|          | 0/800 [00:00<?, ?it/s]


Поиск:   0%|          | 3/800 [00:00<00:30, 26.42it/s]


Поиск:   1%|          | 6/800 [00:00<00:28, 28.36it/s]


Поиск:   1%|          | 9/800 [00:00<00:27, 28.64it/s]

...

,dev Recall@50
geo10_cat0.7,0.860792
char0,0.854542
geo10_cat2,0.852042
geo10_cat0,0.848083
geo5_cat0.7,0.846208
no_quality,0.846208
quality_high,0.844958
geo5_cat2,0.837875
geo5_cat0,0.834750
char2,0.832042


Выбранная конфигурация: geo10_cat0.7


,weight
title,1.50
body,1.00
char,1.00
filters,0.15
geo,10.00
microcat,0.70
quality,0.05


,holdout Recall@50
bm25,0.423750
text,0.395417
selected,0.880000


Bootstrap 95%: [0.8479166666666665, 0.91084375]


## 6. Ошибки

Показываем запросы с полным, частичным и нулевым покрытием позитивов.
Просмотр этих ошибок не меняет параметры стартового решения: конфигурация
уже выбрана на dev. Пропуск сам по себе не доказывает причину ошибки.
Для дальнейшего улучшения можно сравнить позитивные объявления и найденные
кандидаты, затем проверить семантические эмбеддинги на новой оценке.

In [12]:
display(pd.Series({
    'Все позитивы найдены': int((details.recall_at_50 == 1).sum()),
    'Часть позитивов найдена': int(details.recall_at_50.between(0, 1, inclusive='neither').sum()),
    'Нет попаданий': int((details.recall_at_50 == 0).sum()),
}).to_frame('queries'))
display(details.loc[details.recall_at_50 == 0,
                    ['search_query', 'search_location_id', 'search_infm_params_text', 'relevant_count']].head(12))


,queries
Все позитивы найдены,350
Часть позитивов найдена,3
Нет попаданий,47


,search_query,search_location_id,search_infm_params_text,relevant_count
10,master маникюра рязань район рязань,652460,Поиск по слотам Сегодня Поиск по слотам Завтра...,1
19,автободбор,646600,,1
25,автоинструктор по вождению на механике в путил...,637680,"Вид услуги Обучение, курсы",1
140,биохимическая завивка волос,640860,Тип услуги Услуги парикмахера Вид услуги Красо...,1
213,вынести стиральную машину,637640,Вид услуги Вывоз мусора и вторсырья,1
223,гелевые шарики,653240,"Вид услуги Праздники, мероприятия",1
252,декапирование волос,653240,Тип услуги Услуги парикмахера Вид услуги Красо...,1
258,день рождение,637640,,1
259,детейлинг автомобили,657310,,1
346,изготовление самосвальных платформ,633540,,1


## 7. Финальная модель

После фиксации параметров используем полный train для статистик контекста.
Из индекса удаляем объявления вне benchmark: ответ содержит только допустимые
`item_id`. Словарь и IDF сохраняются, включая влияние добавленных train-текстов.

Модель сохраняем через cloudpickle, чтобы определения классов и функций
из этого ноутбука можно было загрузить в отдельном Python-процессе без `src/`.
Затем удаляем модель из памяти и загружаем её обратно перед инференсом.

In [13]:
model.restrict_items(corpus.item_id)
model.fit_history(train, location_metadata)
assert set(model.item_ids) == set(corpus.item_id)
print('Документов в финальном индексе:', len(model.item_ids))

# Кэш — только ускорение токенизации, его содержимое не влияет на предсказания.
_STEM_CACHE.clear()
with gzip.open(MODEL_PATH, 'wb', compresslevel=3) as stream:
    cloudpickle.dump(model, stream, protocol=5)
del model
gc.collect()
with gzip.open(MODEL_PATH, 'rb') as stream:
    model = cloudpickle.load(stream)
print('Модель загружена из', MODEL_PATH)


2026-09-28 18:40:37,494 INFO Статистики контекста: 497673 обучающих пар


Документов в финальном индексе: 189212


Модель загружена из notebook_outputs/model.pkl.gz


## 8. Предсказания для benchmark и answer.csv

Запросы берём из `benchmark_queries.parquet`. Для каждого возвращаем
50 уникальных объявлений из `benchmark_items.parquet`. Проверяем точное
множество query_id, строковые идентификаторы и отсутствие лишнего CSV-индекса.
Повторно читаем записанный файл, чтобы проверить именно результат на диске.

In [14]:
"""Строгая проверка формата сдачи, включая чтение сохранённого CSV."""

import re
from pathlib import Path

import pandas as pd


def validate_submission(answer, queries, corpus_ids):
    if list(answer.columns) != ["query_id", "answer"]:
        raise ValueError("Ожидаются ровно колонки query_id,answer в таком порядке")
    expected = queries.query_id.tolist()
    if len(expected) != len(set(expected)):
        raise ValueError("Повтор query_id во входных запросах")
    if len(answer) != len(expected) or answer.query_id.duplicated().any():
        raise ValueError("Неверное число строк или повторы query_id")
    if set(answer.query_id) != set(expected):
        raise ValueError("Множество query_id не совпадает с benchmark_queries")
    corpus_ids = set(corpus_ids)
    lengths = []
    for qid, raw in answer.itertuples(index=False, name=None):
        if not isinstance(qid, str) or len(qid) != 16:
            raise ValueError(f"Неверный query_id: {qid!r}")
        if not isinstance(raw, str):
            raise ValueError("answer должен быть строкой")
        ids = raw.split(" ") if raw else []
        if not 1 <= len(ids) <= 50 or len(ids) != len(set(ids)):
            raise ValueError(f"Неверное число/повторы item_id для {qid}")
        if any(not re.fullmatch(r"[0-9a-f]{16}", i) for i in ids):
            raise ValueError(f"Неверный формат item_id для {qid}")
        if not set(ids) <= corpus_ids:
            raise ValueError(f"item_id за пределами корпуса для {qid}")
        lengths.append(len(ids))
    return {"rows": len(answer), "min_candidates": min(lengths),
            "max_candidates": max(lengths), "valid": True}


def save_submission(queries, predictions, corpus_ids, output):
    answer = pd.DataFrame({"query_id": queries.query_id,
                           "answer": [" ".join(ids) for ids in predictions]})
    validate_submission(answer, queries, corpus_ids)
    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    answer.to_csv(output, index=False, encoding="utf-8", lineterminator="\n")
    reread = pd.read_csv(output, dtype=str, keep_default_na=False)
    return validate_submission(reread, queries, corpus_ids)


In [15]:
predictions = model.predict(queries)
checks = save_submission(queries, predictions, corpus.item_id, ANSWER_PATH)
answer = pd.read_csv(ANSWER_PATH, dtype=str, keep_default_na=False)
answer_hash = hashlib.sha256(ANSWER_PATH.read_bytes()).hexdigest()
manifest = {
    'seed': SEED,
    'config': model.config.as_dict(),
    'corpus_size': len(corpus),
    'query_count': len(queries),
    'submission_checks': checks,
    'answer_sha256': answer_hash,
    'elapsed_seconds': round(time.perf_counter() - started, 2),
}
(REPORT_DIR / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
print('Проверка:', checks)
print('SHA256:', answer_hash)
print('Файл для отправки:', ANSWER_PATH.resolve())
if previous_answer_hash:
    print('Совпадает с ранее сохранённым answer.csv:', answer_hash == previous_answer_hash)
display(answer.head())



Поиск:   0%|          | 0/2452 [00:00<?, ?it/s]


Поиск:   0%|          | 8/2452 [00:00<00:32, 75.83it/s]


Поиск:   1%|          | 19/2452 [00:00<00:27, 89.68it/s]


Поиск:   1%|          | 30/2452 [00:00<00:25, 93.59it/s]


Поиск:   2%|▏         | 41/2452 [00:00<00:25, 95.49it/s]

...


Поиск:  99%|█████████▉| 2438/2452 [00:24<00:00, 98.59it/s] 


Поиск: 100%|█████████▉| 2448/2452 [00:25<00:00, 97.73it/s]


Поиск: 100%|██████████| 2452/2452 [00:25<00:00, 97.66it/s]

Проверка: {'rows': 2452, 'min_candidates': 50, 'max_candidates': 50, 'valid': True}
SHA256: d93acf026e711a66704764a229785fc3b6fad738963364c2ddb22396d8408d69
Файл для отправки: /private/tmp/avito-notebook-check-sm6j76i3/answer.csv


,query_id,answer
0,70DfDUpwjxB4lzFd,b92ee8f432cec2d1 255fbeaf526a1cc1 355392014208...
1,JTrdTaZJvSiLPkXj,422d3ffdd5bbf626 cbeccbecb1fb8d86 1c6d2079e07f...
2,LZCZNoVG4AFUkVRJ,dab52187b4500d9b 3f89b8062dc85f1c d8fce513e4f0...
3,660ac9QVtXkRxZC3,4c15e7abba063ac5 af91ec4a29b66636 1e5924dc546b...
4,YgHcM9MVbxKnxD1e,1a62634394544781 9c41fe789f98d702 53f9c8caeeb8...


## 9. Повторный инференс и использованные инструменты

Для последующего инференса достаточно сохранённого `notebook_outputs/model.pkl.gz`,
benchmark-запросов и установленных библиотек. Код классов включён в артефакт:

```python
import gzip, cloudpickle, pandas as pd
with gzip.open('notebook_outputs/model.pkl.gz', 'rb') as stream:
    model = cloudpickle.load(stream)
queries = pd.read_parquet('datasets/benchmark_queries.parquet')
predictions = model.predict(queries)
pd.DataFrame({
    'query_id': queries['query_id'],
    'answer': [' '.join(ids) for ids in predictions],
}).to_csv('answer.csv', index=False)
```

Использованы классический BM25 (собственная разреженная реализация),
[scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html),
[NLTK Snowball](https://www.nltk.org/api/nltk.stem.snowball.html), SciPy, NumPy,
pandas, PyArrow, tqdm и cloudpickle. Предобученных нейросетей в этой версии нет.
Параметры, локальные метрики и контрольная сумма CSV сохраняются вместе
с результатами. Скрытая метрика станет известна только после отправки файла.